# VFD Crystallisation — Parameter Estimation Demo

Demonstrates:
1. Synthetic data generation
2. Parameter recovery from known ground truth
3. Bootstrap confidence intervals
4. Profile likelihood
5. Model comparison (AIC/BIC)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from vfd.crystallisation.synthetic_data import (
    ModelParams, ExperimentConfig, generate_vfd_dataset,
    generate_misspecified_dataset, simulate_observables,
)
from vfd.crystallisation.estimation import (
    fit_parameters, bootstrap_parameter_ci, profile_likelihood,
    compare_models, objective_function,
)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Generate Synthetic Data with Known Parameters

In [ ]:
theta_true = ModelParams(alpha=1.0, beta=0.5, gamma=0.8, lam=1.0,
                         eta=0.005, Gamma=0.0, R_c=0.0, kappa_B=1.0, kappa_Q=1.0)
config = ExperimentConfig(n_modes=3, n_trials=200, n_basins=3, max_steps=1000, seed=42)

dataset = generate_vfd_dataset(theta_true, config)
print('Summary:', {k: round(v, 4) if isinstance(v, float) else v
                   for k, v in dataset.summary.items() if k != 'basin_preference'})
print('Basin preference:', dataset.summary['basin_preference'].round(3))

## 2. Fit Parameters

In [ ]:
# Start from a perturbed initial guess
theta_init = ModelParams(alpha=2.0, beta=1.0, gamma=0.3)
fit = fit_parameters(dataset, config, theta_init=theta_init, n_restarts=3)

print(f'Success: {fit.success}')
print(f'Residual: {fit.residual_norm:.6f}')
print(f'Evaluations: {fit.n_evaluations}')
print()
print('Parameter recovery:')
for name, true_val, fit_val in zip(fit.param_names, theta_true.to_vector(), fit.theta_vector):
    print(f'  {name:10s}: true={true_val:.4f}  fitted={fit_val:.4f}')

## 3. Bootstrap Confidence Intervals

In [ ]:
boot = bootstrap_parameter_ci(dataset, config, n_boot=20)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(9)
ax.errorbar(x, boot.point_estimates, 
            yerr=[boot.point_estimates - boot.ci_lower, boot.ci_upper - boot.point_estimates],
            fmt='o', capsize=5, label='Bootstrap estimate')
ax.scatter(x, theta_true.to_vector(), color='red', marker='x', s=100, zorder=5, label='True value')
ax.set_xticks(x)
ax.set_xticklabels(ModelParams.param_names(), rotation=45)
ax.set_ylabel('Parameter value')
ax.set_title('Bootstrap CI vs True Parameters')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Profile Likelihood for Alpha

In [ ]:
grid = np.linspace(0.2, 3.0, 10)
prof = profile_likelihood(dataset, config, 'alpha', grid)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(prof.param_grid, prof.profile_values, 'o-', color='steelblue')
ax.axvline(prof.mle_value, color='red', linestyle='--', label=f'MLE = {prof.mle_value:.2f}')
ax.axvline(theta_true.alpha, color='green', linestyle=':', label=f'True = {theta_true.alpha:.2f}')
ax.axvspan(prof.ci_lower, prof.ci_upper, alpha=0.15, color='blue', label='95% CI')
ax.set_xlabel('alpha')
ax.set_ylabel('Profile loss')
ax.set_title('Profile Likelihood: alpha')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Model Comparison: True vs Wrong vs Misspecified

In [ ]:
theta_wrong = ModelParams(alpha=5.0, beta=5.0, gamma=0.1)
specs = [
    ('VFD (true params)', theta_true, 9),
    ('VFD (wrong params)', theta_wrong, 9),
    ('VFD (fitted)', fit.theta_hat, 9),
]
comp = compare_models(dataset, config, specs)

print('Model comparison results:')
for i, name in enumerate(comp.model_names):
    print(f'  {name:25s}: AIC={comp.aic_scores[i]:.2f}  BIC={comp.bic_scores[i]:.2f}  LL={comp.log_likelihoods[i]:.2f}')
print(f'Preferred: {comp.preferred_model} (by {comp.preferred_by})')